In [9]:
import numpy as np
import pandas as pd

# Import Splits
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

# Import Model
from catboost import CatBoostRegressor

# Import Metrics
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import root_mean_squared_error as rmse

In [10]:
# Read in the data
df = pd.read_csv("PUDF_base1_1q2016_cleaned.csv")

In [11]:
# Create columns for a prolonged flag (>= 30 days), for Number of Codes, and a stratification column
df['PROLONGED'] = [1 if los >= 30 else 0 for los in df["LENGTH_OF_STAY"]]
df['NUM_CODES'] = df.iloc[:, 18:40].sum(axis = 1)
df['strata'] = df['DISCHARGE'].astype(str) + '_' + df['PAT_RURAL'].astype(str) + '_' + df['PROLONGED'].astype(str)

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 671138 entries, 0 to 671137
Data columns (total 47 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   RECORD_ID             671138 non-null  int64  
 1   DISCHARGE             671138 non-null  str    
 2   THCIC_ID              671138 non-null  int64  
 3   TYPE_OF_ADMISSION     671138 non-null  float64
 4   SOURCE_OF_ADMISSION   671138 non-null  str    
 5   PAT_STATE             671138 non-null  str    
 6   PAT_ZIP               671138 non-null  int64  
 7   PAT_COUNTRY           671138 non-null  str    
 8   PAT_COUNTY            671138 non-null  float64
 9   PUBLIC_HEALTH_REGION  671138 non-null  float64
 10  PAT_STATUS            671138 non-null  int64  
 11  SEX_CODE              671138 non-null  str    
 12  RACE                  671138 non-null  int64  
 13  ETHNICITY             671138 non-null  int64  
 14  ADMIT_WEEKDAY         671138 non-null  float64
 15  LENGTH_OF_S

In [12]:
# Create a list of all categorical features (needed for CatBoost)
cat_features = (
    df.columns[11:15].tolist() +
    df.columns[16:17].tolist() +
    df.columns[18:41].tolist() +
    df.columns[42:43].tolist() +
    df.columns[44:45].tolist()
)
cat_features

['SEX_CODE',
 'RACE',
 'ETHNICITY',
 'ADMIT_WEEKDAY',
 'PAT_AGE',
 'CODE_1',
 'CODE_2',
 'CODE_3',
 'CODE_4',
 'CODE_5',
 'CODE_6',
 'CODE_7',
 'CODE_8',
 'CODE_9',
 'CODE_10',
 'CODE_11',
 'CODE_12',
 'CODE_13',
 'CODE_14',
 'CODE_15',
 'CODE_16',
 'CODE_17',
 'CODE_18',
 'CODE_19',
 'CODE_20',
 'CODE_21',
 'CODE_22',
 'PAT_RURAL',
 'PROVIDER_RURAL',
 'PROLONGED']

In [13]:
# Clean Categorical Features to make sure everything is a string and no NaN values
df[cat_features] = (
    df[cat_features]
    .fillna("Missing")
    .astype(str)
)

In [14]:
# Create a list of all features and target, getting rid of columns not used in the model
features = cat_features.copy()
features.extend(['PAT2PROV_DISTANCE','LENGTH_OF_STAY','NUM_CODES'])

In [15]:
# Do a train_test_split on the data, with stratification
X_train, X_test, y_train, y_test = train_test_split(df[features],df.LENGTH_OF_STAY,
                                                    shuffle = True,
                                                    random_state = 2222,
                                                    stratify = df['strata'],
                                                   test_size = .2)

In [16]:
# Create the StratifiedKFold
strata_train = df.loc[X_train.index, 'strata']

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2222
)

In [17]:

rmse_scores = []

# Fit the model for each fold
for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, strata_train)):
    X_tr = X_train.iloc[train_idx].copy()
    X_val = X_train.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    # Instantiate Model
    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=100
    )

    model.fit(X_tr,
        y_tr,
        cat_features = cat_features,
        eval_set = (X_val, y_val),
        use_best_model=True)

    # Find predictions and store rmses  
    preds = model.predict(X_val)
    
    rmse_pred = rmse(y_val,preds)

    rmse_scores.append(rmse_pred)

    print(f"Fold {fold+1}: RMSE = {rmse_pred:.3f}")


0:	learn: 14.1963786	test: 14.7530120	best: 14.7530120 (0)	total: 129ms	remaining: 6m 26s
100:	learn: 2.3739373	test: 1.7668492	best: 1.7668492 (100)	total: 5.82s	remaining: 2m 47s
200:	learn: 0.9179364	test: 1.6107626	best: 1.4824769 (140)	total: 9.75s	remaining: 2m 15s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.482476908
bestIteration = 140

Shrink model to first 141 iterations.
Fold 1: RMSE = 1.482
0:	learn: 14.9413709	test: 11.3608096	best: 11.3608096 (0)	total: 64.2ms	remaining: 3m 12s
100:	learn: 2.1454795	test: 1.0911831	best: 1.0911831 (100)	total: 5.49s	remaining: 2m 37s
200:	learn: 0.9579423	test: 0.6252435	best: 0.6252435 (200)	total: 9.85s	remaining: 2m 17s
300:	learn: 0.5331405	test: 0.3574100	best: 0.3574100 (300)	total: 14.3s	remaining: 2m 8s
400:	learn: 0.3036229	test: 0.2508786	best: 0.2507317 (397)	total: 18.9s	remaining: 2m 2s
500:	learn: 0.2158952	test: 0.2101918	best: 0.2101918 (500)	total: 23.4s	remaining: 1m 56s
600:	learn: 0.1785579	tes